In [1]:
import sys
print(sys.executable)

/Users/Maddalena/Documents/color-in-motion/venv/bin/python


In [1]:
import urllib.request
import os

os.makedirs("../data/raw", exist_ok=True)

files = ["title.basics.tsv.gz", "title.ratings.tsv.gz"]
for f in files:
    url = f"https://datasets.imdbws.com/{f}"
    dest = f"../data/raw/{f}"
    if not os.path.exists(dest):
        print(f"Scarico {f}...")
        urllib.request.urlretrieve(url, dest)
        print(f"  fatto")
    else:
        print(f"{f} gia presente")

Scarico title.basics.tsv.gz...
  fatto
Scarico title.ratings.tsv.gz...
  fatto


In [2]:
import pandas as pd

basics = pd.read_csv(
    "../data/raw/title.basics.tsv.gz",
    sep="\t", na_values="\\N", low_memory=False,
    dtype={"startYear": "str", "runtimeMinutes": "str"}
)
print("Righe totali in basics:", len(basics))
basics.head()

Righe totali in basics: 12676776


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,NaN,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,NaN,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,NaN,1,Short


In [3]:
# Carica i voti
ratings = pd.read_csv(
    "../data/raw/title.ratings.tsv.gz",
    sep="\t", na_values="\\N"
)
print("Righe in ratings:", len(ratings))

# Tieni solo i FILM, con anno e generi presenti
film = basics[
    (basics["titleType"] == "movie")
    & (basics["startYear"].notna())
    & (basics["genres"].notna())
].copy()
print("Film (prima dei voti):", len(film))

# Unisci i voti e converti l'anno in numero
film = film.merge(ratings, on="tconst")
film["startYear"] = film["startYear"].astype(int)
print("Film con voti:", len(film))

film[["primaryTitle", "startYear", "genres", "averageRating", "numVotes"]].head()

Righe in ratings: 1700186
Film (prima dei voti): 568952
Film con voti: 336800


,primaryTitle,startYear,genres,averageRating,numVotes
0,Miss Jerry,1894,Romance,5.3,234
1,The Corbett-Fitzsimmons Fight,1897,"Documentary,News,Sport",5.3,606
2,The Story of the Kelly Gang,1906,"Action,Adventure,Biography",6.0,1081
3,The Prodigal Son,1907,Drama,4.8,40
4,Robbery Under Arms,1907,Drama,3.4,36


In [4]:
# Solo film con almeno 10.000 voti
film = film[film["numVotes"] >= 10000]
print("Film rimasti:", len(film))

# Genere principale = primo genere della lista
film["genere_principale"] = film["genres"].str.split(",").str[0]

# Classifica dei generi
film["genere_principale"].value_counts()

Film rimasti: 12489


genere_principale
Comedy         3084
Action         3038
Drama          2469
Crime          1075
Adventure      1003
Biography       759
Horror          653
Documentary     143
Animation        89
Fantasy          77
Mystery          54
Sci-Fi           14
Thriller         14
Romance           7
Family            4
Western           2
Music             1
Musical           1
Film-Noir         1
History           1
Name: count, dtype: int64

In [5]:
generi_scelti = ["Comedy", "Action", "Drama", "Crime", "Horror"]

# Teniamo solo i film di questi cinque generi
film = film[film["genere_principale"].isin(generi_scelti)]

# Per ogni genere prendiamo gli 80 piu votati
campione = film.groupby("genere_principale").apply(
    lambda g: g.nlargest(80, "numVotes")
).reset_index(drop=True)

print("Totale film nel campione:", len(campione))
campione["genere_principale"].value_counts()

Totale film nel campione: 400


/var/folders/ly/h_kts9d54z92kt0g13ljg0ch0000gp/T/ipykernel_8482/712364008.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  campione = film.groupby("genere_principale").apply(


genere_principale
Action    80
Comedy    80
Crime     80
Drama     80
Horror    80
Name: count, dtype: int64

In [6]:
campione.to_csv("../data/processed/campione_film.csv", index=False)
print("Salvato!")

Salvato!
